In [ ]:
import pandas as pd

# 1. LOAD THE DATA (Using '../' to step out of the script folder first)
df_gdp = pd.read_csv('../data/GDP per capita/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv', skiprows=4)
df_women = pd.read_csv('../data/Internet Usage (Women)/API_IT.NET.USER.FE.ZS_DS2_en_csv_v2_16797/API_IT.NET.USER.FE.ZS_DS2_en_csv_v2_16797.csv', skiprows=4)
df_men = pd.read_csv('../data/Internet Usage (Men)/API_IT.NET.USER.MA.ZS_DS2_en_csv_v2_18514/API_IT.NET.USER.MA.ZS_DS2_en_csv_v2_18514.csv', skiprows=4)
df_total = pd.read_csv('../data/Internet Usage (Population)/API_IT.NET.USER.ZS_DS2_en_csv_v2_21/API_IT.NET.USER.ZS_DS2_en_csv_v2_21.csv', skiprows=4)
# 2. HELPER FUNCTION TO CLEAN AND MELT
def clean_and_melt(df, value_name):
    # World Bank data contains 'Indicator Name' and 'Indicator Code' which we don't need
    # It also contains an empty column at the end usually named 'Unnamed: 68'.
    cols_to_drop = ['Indicator Name', 'Indicator Code']
    cols_to_drop += [c for c in df.columns if 'Unnamed' in c]
    df_clean = df.drop(columns=cols_to_drop, errors='ignore')
    
    # By only specifying id_vars, Pandas will automatically melt ALL remaining year columns
    df_long = pd.melt(df_clean, 
                      id_vars=['Country Name', 'Country Code'], 
                      var_name='Year', 
                      value_name=value_name)
    return df_long

# Apply the function to get our long-format tables
df_gdp_long = clean_and_melt(df_gdp, 'GDP_Per_Capita')
df_women_long = clean_and_melt(df_women, 'Female_Usage_Pct')
df_men_long = clean_and_melt(df_men, 'Male_Usage_Pct')
df_total_long = clean_and_melt(df_total, 'Internet_Usage_Pct')

# 3. MERGE ALL WORLD BANK DATA TOGETHER
clean_df = df_gdp_long.merge(df_women_long, on=['Country Name', 'Country Code', 'Year'])
clean_df = clean_df.merge(df_men_long, on=['Country Name', 'Country Code', 'Year'])
clean_df = clean_df.merge(df_total_long, on=['Country Name', 'Country Code', 'Year'])

# Clean up the Year column (convert to integers) and filter from 2000 onwards
clean_df['Year'] = pd.to_numeric(clean_df['Year'], errors='coerce')
clean_df = clean_df[clean_df['Year'] >= 2000]

# 4. ADD THE COST DATA AND CALCULATE AFFORDABILITY

# Use read_excel and specify the exact name of the tab at the bottom of the Excel file
# (Note: make sure the filename matches exactly what is in your folder)
df_cost = pd.read_excel('../data/worldwide_mobile_data_pricing_data.xlsx', sheet_name='Historical Data')

# THE FIX: Dynamically find the column that contains '2023'
# This ignores weird spaces or special dashes in the Excel header
cost_col = [col for col in df_cost.columns if '2023' in str(col)][0]

# Keep only the Name and the specific year's price, then rename it
df_cost = df_cost[['Name', cost_col]].rename(columns={cost_col: 'Cost_1GB_USD'})

# Merge it with our World Bank data
clean_df = clean_df.merge(df_cost, left_on='Country Name', right_on='Name', how='left')
clean_df.drop(columns=['Name'], errors='ignore', inplace=True)

# --- NEW CLEANING STEP ---
# 1. Convert the column to text, strip out any '$' or commas
clean_df['Cost_1GB_USD'] = clean_df['Cost_1GB_USD'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)

# 2. Force both columns to be numeric (errors='coerce' turns text like "N/A" into blank NaN values)
clean_df['Cost_1GB_USD'] = pd.to_numeric(clean_df['Cost_1GB_USD'], errors='coerce')
clean_df['GDP_Per_Capita'] = pd.to_numeric(clean_df['GDP_Per_Capita'], errors='coerce')


# Calculate the UN 2% Affordability Metric
clean_df['Affordability_Pct'] = ((clean_df['Cost_1GB_USD'] * 12) / clean_df['GDP_Per_Capita']) * 100



# 5. EXPORT THE FINAL DASHBOARD FILE
# Drop completely empty rows so Altair runs smoothly
clean_df.dropna(subset=['GDP_Per_Capita', 'Internet_Usage_Pct'], how='all', inplace=True)

clean_df.to_csv('cleaned_dashboard_data.csv', index=False)
print("Data successfully cleaned and saved to 'cleaned_dashboard_data.csv'!")

Data successfully cleaned and saved to 'cleaned_dashboard_data.csv'!
Data successfully cleaned and saved to 'cleaned_dashboard_data.csv'!


In [12]:
# Calculate the percentage of missing values per column
missing_pct = clean_df.isna().mean() * 100

# Filter out columns with 0% missing, and sort the rest from highest to lowest
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

# Format the output to look like a clean percentage
print("Percentage of missing values per column:")
print(missing_pct.round(2).astype(str) + '%')

Percentage of missing values per column:
Female_Usage_Pct      77.28%
Male_Usage_Pct        77.25%
Affordability_Pct      32.8%
Cost_1GB_USD          32.37%
Internet_Usage_Pct    16.53%
GDP_Per_Capita         1.44%
dtype: object


In [13]:
!pip install pycountry streamlit altair pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 10.3 MB/s eta 0:00:0000:0100:01
